# chunk_500 title -> 질문형 변환

본문에 물음표(질문)가 있는 청크는 그대로 추출해서 title로 쓰고,
없는 청크는 Qwen2.5-1.5B-Instruct로 질문을 생성한다.

실행 전에: 런타임 > 런타임 유형 변경 > GPU(T4) 선택. `chunks_500.json`은 아래 업로드 셀에서 직접 선택해서 올린다.

In [ ]:
!pip install -q transformers accelerate

## 0) chunks_500.json 업로드
아래 셀을 실행하면 파일 선택 창이 뜬다. `chunks_500.json`을 선택하면 현재 작업 폴더에 업로드된다.

In [ ]:
from google.colab import files

uploaded = files.upload()
print("업로드된 파일:", list(uploaded.keys()))

In [ ]:
import json
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

CHUNKS_PATH = "chunks_500.json"   # 위 업로드 셀에서 선택한 파일명과 같아야 함
OUT_PATH    = "chunks_500_qtitle.json"
MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"  # GPU 있으니 0.5B보다 큰 모델 사용
BATCH_SIZE  = 32

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(device)
model.eval()

In [ ]:
data = json.load(open(CHUNKS_PATH, encoding="utf-8"))
print("전체 청크:", len(data))

## 1) 본문 안에 이미 있는 질문 추출 (물음표 기준)

In [ ]:
Q_RE = re.compile(r"[^.!?]*\?")

def extract_question(text):
    matches = Q_RE.findall(text)
    if not matches:
        return None
    q = matches[0].strip()
    if len(q) < 8 or len(q) > 250:   # 너무 짧거나(파편) 너무 길면(여러 문장 뭉침) 버림 -> LLM 생성으로 넘김
        return None
    return q

need_gen = []
for i, d in enumerate(data):
    q = extract_question(d["text"])
    if q:
        d["title"] = q
    else:
        need_gen.append(i)

print(f"본문에서 질문 추출: {len(data) - len(need_gen)}개")
print(f"LLM 생성 필요: {len(need_gen)}개")

## 2) 질문이 없는 청크는 Qwen으로 생성 (배치 처리)

In [ ]:
def build_prompt(text):
    messages = [{
        "role": "user",
        "content": f"다음 글의 내용을 한 문장의 영어 질문으로 만들어줘. 질문만 출력하고 다른 설명은 하지마:\n\n{text[:600]}",
    }]
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

for start in range(0, len(need_gen), BATCH_SIZE):
    batch_idx = need_gen[start:start + BATCH_SIZE]
    prompts = [build_prompt(data[i]["text"]) for i in batch_idx]
    inputs = tok(prompts, return_tensors="pt", padding=True).to(device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40, do_sample=False)

    for j, i in enumerate(batch_idx):
        gen_ids = out[j][inputs["input_ids"].shape[1]:]
        q = tok.decode(gen_ids, skip_special_tokens=True).strip().split("\n")[0].strip()
        if q:
            data[i]["title"] = q
        # 생성 실패 시 기존 title(원래 TREC 쿼리)을 그대로 둠 -> 데이터 유실 없음

    done = start + len(batch_idx)
    if done % (BATCH_SIZE * 5) == 0 or done == len(need_gen):
        print(f"{done}/{len(need_gen)} 처리")

## 3) 저장

In [ ]:
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("완료:", OUT_PATH)